# Lab: Hồi quy Tuyến tính (Linear Regression)

## 1. Bài toán hồi quy là gì?

Khác với **phân loại** (classification) — đoán nhãn rời rạc, **hồi quy** (regression) đoán giá trị *liên tục*: giá nhà, nhiệt độ, doanh thu...

Ví dụ: cho diện tích nhà → đoán giá. Mỗi diện tích cho ra một số thực, không phải nhóm.

## 2. Mô hình tuyến tính

Mô hình đơn giản nhất: giả sử mối quan hệ giữa input $x = (x_1, x_2, \dots, x_n)$ và output $y$ là **tuyến tính**:
$$
\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = w^T x + b
$$

Mục tiêu: tìm $w$ và $b$ sao cho $\hat{y}$ gần $y$ thực tế nhất.

## 3. Hàm mất mát: Mean Squared Error

$$
L(w, b) = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2 = \frac{1}{N}\sum_{i=1}^{N}(w^T x_i + b - y_i)^2
$$

MSE phạt **bình phương** sai số → outlier ảnh hưởng rất mạnh. Nếu data có nhiều outlier, dùng **MAE** (Mean Absolute Error) hoặc **Huber loss** thay thế.

## 4. Hai cách tìm $(w, b)$

### 4.1. Normal Equation — công thức đóng (closed-form)

Đặt $X$ là ma trận $N \times (n+1)$ (thêm cột 1 cho bias), $y$ là vector $N \times 1$. Giải $\nabla L = 0$ cho ra:
$$
\hat{\theta} = (X^T X)^{-1} X^T y
$$

Đẹp, không cần lặp. Nhược: $O(n^3)$ để inverse → chậm khi nhiều feature.

### 4.2. Gradient Descent — lặp

Gradient của MSE:
$$\frac{\partial L}{\partial w_j} = \frac{2}{N}\sum_{i=1}^{N}(w^T x_i + b - y_i) \cdot x_{i,j}$$

Cập nhật:
$$w \leftarrow w - \alpha \nabla_w L, \quad b \leftarrow b - \alpha \frac{\partial L}{\partial b}$$

Chậm hơn về toán nhưng scale được lên dataset cực lớn. Đây là cách *neural network* học.

## 5. Đánh giá: $R^2$ và RMSE

**RMSE** (Root MSE): $\sqrt{L}$ — đơn vị = đơn vị của $y$ → dễ diễn giải.

**$R^2$ (coefficient of determination)**: tỷ lệ variance của $y$ được model giải thích.
$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
$R^2 = 1$: hoàn hảo. $R^2 = 0$: bằng đoán mean. $R^2 < 0$: tệ hơn đoán mean.

# THỰC HÀNH 1: Hồi quy 1 biến với dữ liệu giả lập

Sinh dữ liệu $y = 3x + 5 + \varepsilon$, train cả 3 cách (Normal Equation, GD thủ công, sklearn) và so sánh.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline

np.random.seed(42)

# Sinh data y = 3x + 5 + nhiễu
N = 100
x = np.linspace(0, 10, N)
y = 3 * x + 5 + np.random.randn(N) * 1.5    # nhiễu N(0, 1.5)

plt.figure(figsize=(8, 4))
plt.scatter(x, y, alpha=0.6, label='Dữ liệu')
plt.plot(x, 3*x + 5, 'r-', label='True line: y = 3x + 5')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Dữ liệu giả lập')
plt.show()

### Cách 1: Normal Equation

In [ ]:
# Thêm cột 1 cho bias: X có shape (N, 2), cột đầu là 1, cột sau là x.
X = np.column_stack([np.ones(N), x])
theta_ne = np.linalg.inv(X.T @ X) @ X.T @ y
b_ne, w_ne = theta_ne
print(f'Normal Equation: w = {w_ne:.4f},  b = {b_ne:.4f}')
print(f'                 (so với true: w=3, b=5)')

### Cách 2: Gradient Descent thủ công

In [ ]:
w, b = 0.0, 0.0
lr = 0.01
loss_history = []

for step in range(1000):
    y_hat = w * x + b
    err = y_hat - y
    loss = (err ** 2).mean()
    loss_history.append(loss)

    # gradient
    dw = 2 * (err * x).mean()
    db = 2 * err.mean()
    w -= lr * dw
    b -= lr * db

print(f'Gradient Descent: w = {w:.4f},  b = {b:.4f}')
print(f'Final loss: {loss:.4f}')

plt.figure(figsize=(8, 3.5))
plt.plot(loss_history); plt.xlabel('Step'); plt.ylabel('MSE'); plt.yscale('log')
plt.title('Loss giảm theo số bước GD'); plt.grid(alpha=0.3); plt.show()

### Cách 3: sklearn

In [ ]:
model = LinearRegression()
model.fit(x.reshape(-1, 1), y)
print(f'sklearn:         w = {model.coef_[0]:.4f},  b = {model.intercept_:.4f}')

# Vẽ ba đường
plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.5, label='Data')
plt.plot(x, 3*x + 5, 'r-', linewidth=2, label='True')
plt.plot(x, w_ne*x + b_ne, 'g--', linewidth=2, label='Normal Equation')
plt.plot(x, w*x + b, 'b:', linewidth=2, label='Gradient Descent')
plt.plot(x, model.coef_[0]*x + model.intercept_, 'm-.', linewidth=2, label='sklearn')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print('Cả 3 cách cho ra gần như cùng kết quả (chỉ lệch nhỏ ở GD nếu chưa đủ vòng).')

# THỰC HÀNH 2: Hồi quy đa biến — California Housing

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X, y = housing.data, housing.target
feat_names = housing.feature_names
print(f'Shape: X={X.shape}, y={y.shape}')
print(f'Features: {feat_names}')
print(f'Target = giá nhà (đơn vị: trăm nghìn USD)')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scaling: fit chỉ trên train để tránh data leakage.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
print(f'Test RMSE: {rmse:.3f}  (đơn vị $100K)')
print(f'Test MAE : {mae:.3f}')
print(f'Test R^2 : {r2:.3f}')

# Hệ số
coef = pd.Series(model.coef_, index=feat_names).sort_values()
coef.plot.barh(figsize=(7, 4)); plt.xlabel('Hệ số (đã chuẩn hoá)')
plt.title('Tác động của từng feature lên giá nhà'); plt.tight_layout(); plt.show()

In [ ]:
# Vẽ predicted vs actual
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Hoàn hảo')
plt.xlabel('Giá thật'); plt.ylabel('Giá dự đoán')
plt.title(f'Predicted vs Actual (R² = {r2:.3f})')
plt.legend(); plt.grid(alpha=0.3)
plt.axis('equal'); plt.show()

## 6. Regularization — Ridge và Lasso

Hồi quy thường gặp vấn đề khi feature có tương quan cao (multicollinearity) → hệ số bùng nổ. **Regularization** thêm penalty vào loss để khống chế:

**Ridge (L2)**: $L = \text{MSE} + \alpha \sum w_j^2$. Co hệ số về 0 nhưng không chính xác là 0.

**Lasso (L1)**: $L = \text{MSE} + \alpha \sum |w_j|$. Có thể đẩy hệ số về *đúng 0* → tự động chọn feature.

In [ ]:
for name, mdl in [('Linear', LinearRegression()),
                  ('Ridge α=1', Ridge(alpha=1.0)),
                  ('Ridge α=10', Ridge(alpha=10.0)),
                  ('Lasso α=0.1', Lasso(alpha=0.1)),
                  ('Lasso α=1', Lasso(alpha=1.0))]:
    mdl.fit(X_train_s, y_train)
    pred = mdl.predict(X_test_s)
    r2 = r2_score(y_test, pred)
    nz = np.sum(np.abs(mdl.coef_) > 1e-6)
    print(f'{name:15s}  R² = {r2:.4f}  | Số coef khác 0: {nz}/{len(mdl.coef_)}')

## 7. Polynomial regression — học quan hệ phi tuyến

Linear Regression không *bắt buộc* dữ liệu phải tuyến tính theo $x$. Ta có thể tạo feature mới $x^2, x^3$ rồi chạy linear regression — đây là **polynomial regression**.

In [ ]:
# Sinh data phi tuyến: y = sin(x) + nhiễu
x_nl = np.sort(np.random.uniform(0, 2*np.pi, 100))
y_nl = np.sin(x_nl) + np.random.randn(100) * 0.15

plt.figure(figsize=(10, 4))
plt.scatter(x_nl, y_nl, alpha=0.5, label='Data')
for deg in [1, 3, 9]:
    mdl = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    mdl.fit(x_nl.reshape(-1, 1), y_nl)
    xx = np.linspace(0, 2*np.pi, 200).reshape(-1, 1)
    plt.plot(xx, mdl.predict(xx), label=f'degree={deg}', linewidth=2)
plt.legend(); plt.grid(alpha=0.3); plt.title('Polynomial regression bậc khác nhau')
plt.show()
print('degree=1: underfit (đường thẳng)')
print('degree=3: vừa (gần với sin trong khoảng dữ liệu)')
print('degree=9: bắt đầu lượn theo nhiễu — overfit')

## Tổng kết

1. Linear Regression: $\hat{y} = w^T x + b$, tối thiểu MSE.
2. Hai cách giải: Normal Equation (đóng) và Gradient Descent (lặp).
3. Đánh giá: RMSE, MAE, R².
4. **Ridge / Lasso** giúp chống overfitting và (Lasso) chọn feature.
5. **Polynomial regression**: thêm $x^2, x^3, ...$ để học quan hệ phi tuyến — nhưng coi chừng overfit.
6. **Phải scale feature** trước khi dùng regularization (vì penalty so với độ lớn coef).

# BÀI TẬP VỀ NHÀ

## Bài 1: Cài đặt từ scratch
Viết class `MyLinearRegression` với 2 method:
- `fit(X, y)` dùng Normal Equation.
- `predict(X)` dùng coefficient đã fit.

Test trên California Housing. So sánh hệ số và R² với `sklearn.LinearRegression`. Phải gần như giống hệt.

*Gợi ý:* nhớ thêm cột 1 cho bias trước khi inverse.

## Bài 2: Diện tích → giá nhà
Sinh dữ liệu giả lập:
- 200 căn nhà.
- Diện tích: từ 30m² đến 200m².
- Giá = 50 + 30·area + nhiễu (đơn vị triệu).

Train Linear Regression. Vẽ scatter + đường hồi quy. Báo cáo RMSE.

## Bài 3: Outlier ảnh hưởng MSE thế nào?
Lấy data `y = 3x + 5 + nhiễu` ở Lab 1. Thêm 5 outlier mạnh (y = 100). Train Linear Regression.

Sau đó dùng `HuberRegressor` (`from sklearn.linear_model import HuberRegressor`) — robust với outlier. So sánh hệ số học được. Cái nào sát true (3, 5) hơn?

## Bài 4: Ridge vs Lasso khi có feature thừa
Sinh dữ liệu: 100 mẫu, 20 feature, nhưng *chỉ 5 feature thật sự liên quan* đến y (15 còn lại là nhiễu).

1. Train `LinearRegression`, `Ridge(α=1)`, `Lasso(α=0.1)`.
2. Đếm số coef khác 0 sau train.
3. Lasso có "phát hiện" được 5 feature thật không?

*Gợi ý:* dùng `np.abs(model.coef_) > 1e-6` để đếm.

## Bài 5: Polynomial degree → overfit?
Trên data sin ở Lab cuối cùng:
1. Train polynomial regression với `degree ∈ {1, 2, ..., 15}`.
2. Với mỗi degree, chia 80/20 train/test, đo RMSE trên cả train và test.
3. Vẽ 2 đường RMSE theo degree.
4. Quan sát: train RMSE giảm liên tục, test RMSE giảm rồi tăng — đó là *bias-variance trade-off*. Tại degree nào test RMSE thấp nhất?